In [2]:
!pip install -q -U ultralytics roboflow

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("sohamblulabel")  # re-add this secret if it's a fresh notebook

import torch
import ultralytics
from ultralytics import YOLO
from roboflow import Roboflow

ultralytics.checks()

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Notebook Settings → Accelerator → GPU T4 x2, then restart."
    )
print(f"GPU available: {torch.cuda.get_device_name(0)}")

rf = Roboflow(api_key=secret_value_0)
project = rf.workspace("soham-bhattacharya-mwcvr").project("bluarmor-od-wd3wq")
version = project.version(3)
dataset = version.download("yolov11")
print(f"\nDataset downloaded to: {dataset.location}")

Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.6/8062.4 GB disk)
GPU available: Tesla T4
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to BluArmor-OD-3 in yolov11:: 100%|██████████| 2325/2325 [00:00<00:00, 12068.50it/s]


Dataset downloaded to: /kaggle/working/BluArmor-OD-3


In [3]:
import os
import yaml

DATASET_PATH = dataset.location
data_yaml_path = os.path.join(DATASET_PATH, "data.yaml")

with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

fixed_config = dict(data_config)
for split_key, folder_name in [("train", "train"), ("val", "valid"), ("test", "test")]:
    img_dir = os.path.join(DATASET_PATH, folder_name, "images")
    if os.path.isdir(img_dir):
        fixed_config[split_key] = img_dir
fixed_config["path"] = DATASET_PATH

fixed_yaml_path = "/kaggle/working/data.yaml"
with open(fixed_yaml_path, "w") as f:
    yaml.dump(fixed_config, f, default_flow_style=False)

class_names = fixed_config.get("names", [])
print(f"Classes detected: {class_names}")

Classes detected: ['defective', 'passing']


In [4]:
from collections import Counter

def gather_pairs(img_dir):
    label_dir = img_dir.replace("images", "labels")
    pairs = []
    for fname in sorted(os.listdir(img_dir)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        img_path = os.path.join(img_dir, fname)
        lbl_path = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")
        pairs.append((img_path, lbl_path, fname))
    return pairs

all_pairs = []
for split_key in ["train", "val", "test"]:
    if split_key in fixed_config:
        all_pairs.extend(gather_pairs(fixed_config[split_key]))
print(f"Total images pooled: {len(all_pairs)}")

def get_class_for_image(lbl_path):
    """Assumes one connector class per image, per this project's labeling scheme."""
    if not os.path.exists(lbl_path) or os.path.getsize(lbl_path) == 0:
        return None
    with open(lbl_path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    if not lines:
        return None
    class_idx = int(lines[0].split()[0])
    return class_names[class_idx] if 0 <= class_idx < len(class_names) else None

def get_color_for_image(fname):
    """ADJUST THIS if your white images used a different naming convention —
    verify against the printed counts below before trusting it."""
    return "white" if "white" in fname.lower() else "black"

tagged_pairs = []
for img_path, lbl_path, fname in all_pairs:
    cls = get_class_for_image(lbl_path)
    if cls is None:
        continue
    tagged_pairs.append((img_path, lbl_path, cls, get_color_for_image(fname)))

group_counts = Counter((cls, color) for _, _, cls, color in tagged_pairs)
print("Group counts (class, color) — verify these match expectations:")
for group, count in sorted(group_counts.items()):
    print(f"  {group}: {count}")

Total images pooled: 1160
Group counts (class, color) — verify these match expectations:
  ('defective', 'black'): 477
  ('passing', 'black'): 683


In [5]:
from sklearn.model_selection import train_test_split

group_key = [f"{cls}_{color}" for _, _, cls, color in tagged_pairs]

train_pairs, temp_pairs = train_test_split(
    tagged_pairs, test_size=0.30, stratify=group_key, random_state=42
)
temp_group_key = [f"{cls}_{color}" for _, _, cls, color in temp_pairs]
test_pairs, val_pairs = train_test_split(
    temp_pairs, test_size=(10/30), stratify=temp_group_key, random_state=42
)

print(f"Train: {len(train_pairs)} | Test (20%): {len(test_pairs)} | Val (10%): {len(val_pairs)}")
for name, pairs in [("train", train_pairs), ("test", test_pairs), ("val", val_pairs)]:
    counts = Counter((c, col) for _, _, c, col in pairs)
    print(f"  {name}: {dict(counts)}")

Train: 812 | Test (20%): 232 | Val (10%): 116
  train: {('passing', 'black'): 478, ('defective', 'black'): 334}
  test: {('passing', 'black'): 137, ('defective', 'black'): 95}
  val: {('defective', 'black'): 48, ('passing', 'black'): 68}


In [6]:
import cv2
import shutil

ZOOMED_ROOT = "/kaggle/working/zoomed_dataset"
ZOOM_FACTOR = 2.0
MAX_DIM = 1280
MIN_BOX_FRACTION = 0.15

if os.path.exists(ZOOMED_ROOT):
    shutil.rmtree(ZOOMED_ROOT)  # clear stale output — this is what fixed the v3 leakage incident

def adjust_and_write_labels(lbl_path, out_lbl_path, orig_w, orig_h, crop_w, crop_h, x0, y0, min_frac):
    if not os.path.exists(lbl_path):
        open(out_lbl_path, "w").close()
        return
    with open(lbl_path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    out_lines = []
    for line in lines:
        parts = line.split()
        cls_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])

        abs_xc, abs_yc = xc * orig_w, yc * orig_h
        abs_bw, abs_bh = bw * orig_w, bh * orig_h
        x1, y1 = abs_xc - abs_bw / 2, abs_yc - abs_bh / 2
        x2, y2 = abs_xc + abs_bw / 2, abs_yc + abs_bh / 2

        orig_area = (x2 - x1) * (y2 - y1)
        x1c, y1c = max(x1 - x0, 0), max(y1 - y0, 0)
        x2c, y2c = min(x2 - x0, crop_w), min(y2 - y0, crop_h)

        if x2c <= x1c or y2c <= y1c:
            continue
        if (x2c - x1c) * (y2c - y1c) / orig_area < min_frac:
            continue

        new_bw, new_bh = (x2c - x1c) / crop_w, (y2c - y1c) / crop_h
        new_xc, new_yc = (x1c + x2c) / 2 / crop_w, (y1c + y2c) / 2 / crop_h
        out_lines.append(f"{cls_id} {new_xc:.6f} {new_yc:.6f} {new_bw:.6f} {new_bh:.6f}")

    with open(out_lbl_path, "w") as f:
        f.write("\n".join(out_lines))

for split_name, pairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
    img_out_dir = os.path.join(ZOOMED_ROOT, split_name, "images")
    lbl_out_dir = os.path.join(ZOOMED_ROOT, split_name, "labels")
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    for img_path, lbl_path, cls, color in sorted(pairs, key=lambda p: p[0]):
        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        crop_w, crop_h = w / ZOOM_FACTOR, h / ZOOM_FACTOR
        x0, y0 = (w - crop_w) / 2, (h - crop_h) / 2
        cropped = img[int(y0):int(y0 + crop_h), int(x0):int(x0 + crop_w)]

        scale = min(1.0, MAX_DIM / max(cropped.shape[:2]))
        if scale < 1.0:
            cropped = cv2.resize(cropped, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

        base_name = os.path.splitext(os.path.basename(img_path))[0]
        out_name = f"{color}__{base_name}"
        cv2.imwrite(os.path.join(img_out_dir, out_name + ".jpg"), cropped)
        adjust_and_write_labels(
            lbl_path, os.path.join(lbl_out_dir, out_name + ".txt"),
            w, h, crop_w, crop_h, x0, y0, MIN_BOX_FRACTION
        )

zoomed_yaml_path = "/kaggle/working/zoomed_data.yaml"
zoomed_config = {
    "path": ZOOMED_ROOT,
    "train": os.path.join(ZOOMED_ROOT, "train", "images"),
    "val": os.path.join(ZOOMED_ROOT, "val", "images"),
    "test": os.path.join(ZOOMED_ROOT, "test", "images"),
    "nc": len(class_names),
    "names": class_names,
}
with open(zoomed_yaml_path, "w") as f:
    yaml.dump(zoomed_config, f, default_flow_style=False)
print(f"Zoomed dataset written to: {zoomed_yaml_path}")

Zoomed dataset written to: /kaggle/working/zoomed_data.yaml


In [7]:
import albumentations as A

augment_pipeline = A.Compose([
    A.Rotate(limit=8, border_mode=cv2.BORDER_REPLICATE, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.7),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10, p=0.5),
    A.GaussNoise(std_range=(0.01, 0.03), p=0.3),
    A.GaussianBlur(blur_limit=(3, 3), p=0.2),
], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.4))

train_img_dir = os.path.join(ZOOMED_ROOT, "train", "images")
train_lbl_dir = os.path.join(ZOOMED_ROOT, "train", "labels")

for fname in list(os.listdir(train_img_dir)):
    if "_aug" in fname:
        os.remove(os.path.join(train_img_dir, fname))
for fname in list(os.listdir(train_lbl_dir)):
    if "_aug" in fname:
        os.remove(os.path.join(train_lbl_dir, fname))

base_images = sorted(f for f in os.listdir(train_img_dir) if "_aug" not in f)

for fname in base_images:
    base_name = os.path.splitext(fname)[0]
    img = cv2.imread(os.path.join(train_img_dir, fname))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    lbl_path = os.path.join(train_lbl_dir, base_name + ".txt")
    bboxes, class_labels = [], []
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                if line.strip():
                    parts = line.split()
                    class_labels.append(int(float(parts[0])))
                    bboxes.append([float(x) for x in parts[1:5]])

    for aug_idx in range(2):
        augmented = augment_pipeline(image=img_rgb, bboxes=bboxes, class_labels=class_labels)
        aug_img = cv2.cvtColor(augmented["image"], cv2.COLOR_RGB2BGR)
        out_name = f"{base_name}_aug{aug_idx}"
        cv2.imwrite(os.path.join(train_img_dir, out_name + ".jpg"), aug_img)
        with open(os.path.join(train_lbl_dir, out_name + ".txt"), "w") as f:
            for cls, box in zip(augmented["class_labels"], augmented["bboxes"]):
                f.write(f"{int(cls)} {' '.join(f'{v:.6f}' for v in box)}\n")

print(f"Train images after 3x augmentation: {len(os.listdir(train_img_dir))}")

Train images after 3x augmentation: 2436


In [8]:
import re

def get_source_id(filename):
    name = os.path.splitext(filename)[0]
    return re.sub(r"_aug\d+$", "", name)

split_ids = {
    split: set(get_source_id(f) for f in os.listdir(os.path.join(ZOOMED_ROOT, split, "images")))
    for split in ["train", "val", "test"]
}
overlaps = {f"{a}/{b}": split_ids[a] & split_ids[b] for a, b in [("train","val"),("train","test"),("val","test")]}

if any(overlaps.values()):
    for pair, ov in overlaps.items():
        if ov:
            print(f"LEAKAGE in {pair}: {len(ov)} overlapping images")
    raise RuntimeError("DATA LEAKAGE DETECTED — refusing to proceed to training.")
print("No leakage detected — safe to proceed to training.")

No leakage detected — safe to proceed to training.


In [16]:
from ultralytics import YOLO

# Upload your current aoi_pass_defect_v4/best.pt as a Kaggle input dataset first
# (Add Data → Upload), then set this path to match.
PREVIOUS_BEST_PATH = "/kaggle/input/models/sohambhattacharya123/yolov11n/pytorch/default/1/best.pt"

model = YOLO(PREVIOUS_BEST_PATH)
results = model.train(
    data=zoomed_yaml_path,
    imgsz=640,
    epochs=40,
    batch=16,
    patience=15,
    lr0=0.002,        # lower than a from-scratch run — this is a fine-tune
    device=0,
    project="/kaggle/working/runs",
    name="aoi_pass_defect_v6_multicolor",
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    seed=42,
    fliplr=0.0,
)
print("Best weights: /kaggle/working/runs/aoi_pass_defect_v6_multicolor/weights/best.pt")

Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/zoomed_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/models/sohambhattacharya123/yolov11n/pytorch/default/1/best.pt, momentum=0.9

In [17]:
best_model = YOLO("/kaggle/working/runs/aoi_pass_defect_v6_multicolor/weights/best.pt")

test_metrics = best_model.val(data=zoomed_yaml_path, imgsz=640, device=0, split="test")
print("=== Overall Test Metrics ===")
print(f"mAP50: {test_metrics.box.map50:.4f} | mAP50-95: {test_metrics.box.map:.4f}")
for i, cname in enumerate(class_names):
    print(f"  {cname}: AP50 = {test_metrics.box.ap50[i]:.4f}")

def build_color_subset_yaml(color):
    src_img_dir = os.path.join(ZOOMED_ROOT, "test", "images")
    src_lbl_dir = os.path.join(ZOOMED_ROOT, "test", "labels")
    subset_root = f"/kaggle/working/test_{color}_only"
    if os.path.exists(subset_root):
        shutil.rmtree(subset_root)
    img_out, lbl_out = os.path.join(subset_root, "images"), os.path.join(subset_root, "labels")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)
    for fname in os.listdir(src_img_dir):
        if fname.lower().startswith(f"{color}__"):
            shutil.copy(os.path.join(src_img_dir, fname), img_out)
            lbl_src = os.path.join(src_lbl_dir, os.path.splitext(fname)[0] + ".txt")
            if os.path.exists(lbl_src):
                shutil.copy(lbl_src, lbl_out)
    subset_yaml = f"{subset_root}.yaml"
    with open(subset_yaml, "w") as f:
        yaml.dump({"path": subset_root, "train": img_out, "val": img_out, "test": img_out,
                   "nc": len(class_names), "names": class_names}, f)
    return subset_yaml, img_out

for color in ["black", "white"]:
    subset_yaml, img_out = build_color_subset_yaml(color)
    n = len(os.listdir(img_out))
    print(f"\n=== {color.upper()}-only test subset ({n} images) ===")
    if n == 0:
        print("  No images found — recheck the color heuristic in Cell 2a.")
        continue
    m = best_model.val(data=subset_yaml, imgsz=640, device=0, split="test")
    print(f"mAP50: {m.box.map50:.4f}")
    for i, cname in enumerate(class_names):
        if i < len(m.box.ap50):
            print(f"  {cname}: AP50 = {m.box.ap50[i]:.4f}")

Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 414.8±119.1 MB/s, size: 11.8 KB)
val: Scanning /kaggle/working/zoomed_dataset/test/labels... 232 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 232/232 1.3Kit/s 0.2s<0.3s
val: New cache created: /kaggle/working/zoomed_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 5.4it/s 2.8s0.1s
                   all        232        232      0.999      0.995       0.99      0.723
             defective         95         95      0.999      0.989      0.985      0.712
               passing        137        137          1          1      0.995      0.734
Speed: 2.0ms preprocess, 4.4ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-13
==

In [21]:
import re
from collections import Counter

prefix_counts = Counter()
for split_key in ["train", "val", "test"]:
    for fname in os.listdir(fixed_config[split_key]):
        m = re.match(r'^([a-zA-Z]+(?:_[a-zA-Z]+)?)_\d+', fname)
        prefix = m.group(1) if m else "NO_MATCH"
        prefix_counts[prefix] += 1

print(prefix_counts)

Counter({'pass_board': 378, 'def_board': 200, 'pass': 125, 'def_boardaddbatch': 121, 'defect': 101, 'pass_boardadd': 96, 'pass_boardaddbatch': 84, 'def_boardadd': 55})


In [24]:
def get_color_for_image(fname):
    base = fname.lower()
    if re.match(r'^(pass|defect)_\d+', base):
        return "white"
    return "black"

In [25]:
def true_color_from_zoomed_filename(zoomed_fname):
    stripped = zoomed_fname.split("__", 1)[1] if "__" in zoomed_fname else zoomed_fname
    return get_color_for_image(stripped)

def build_color_subset_yaml_fixed(color):
    src_img_dir = os.path.join(ZOOMED_ROOT, "test", "images")
    src_lbl_dir = os.path.join(ZOOMED_ROOT, "test", "labels")
    subset_root = f"/kaggle/working/test_{color}_only_fixed"
    if os.path.exists(subset_root):
        shutil.rmtree(subset_root)
    img_out, lbl_out = os.path.join(subset_root, "images"), os.path.join(subset_root, "labels")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)
    for fname in os.listdir(src_img_dir):
        if true_color_from_zoomed_filename(fname) == color:
            shutil.copy(os.path.join(src_img_dir, fname), img_out)
            lbl_src = os.path.join(src_lbl_dir, os.path.splitext(fname)[0] + ".txt")
            if os.path.exists(lbl_src):
                shutil.copy(lbl_src, lbl_out)
    subset_yaml = f"{subset_root}.yaml"
    with open(subset_yaml, "w") as f:
        yaml.dump({"path": subset_root, "train": img_out, "val": img_out, "test": img_out,
                   "nc": len(class_names), "names": class_names}, f)
    return subset_yaml, img_out

for color in ["black", "white"]:
    subset_yaml, img_out = build_color_subset_yaml_fixed(color)
    n = len(os.listdir(img_out))
    print(f"\n=== {color.upper()}-only test subset ({n} images) ===")
    if n == 0:
        continue
    m = best_model.val(data=subset_yaml, imgsz=640, device=0, split="test")
    print(f"mAP50: {m.box.map50:.4f}")
    for i, cname in enumerate(class_names):
        print(f"  {cname}: AP50 = {m.box.ap50[i]:.4f}")


=== BLACK-only test subset (186 images) ===
Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 464.4±121.0 MB/s, size: 12.2 KB)
val: Scanning /kaggle/working/test_black_only_fixed/labels... 186 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 186/186 1.3Kit/s 0.1s<0.1s
val: New cache created: /kaggle/working/test_black_only_fixed/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 4.6it/s 2.6s0.1s
                   all        186        186      0.999      0.993       0.99      0.677
             defective         71         71      0.999      0.986      0.985      0.644
               passing        115        115      0.999          1      0.995       0.71
Speed: 2.4ms preprocess, 4.4ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-15
mAP50: 0.9900
  defective: AP50 = 0.

In [26]:
import shutil
output_dir = "/kaggle/working/runs/aoi_pass_defect_v6_multicolor"
shutil.make_archive("/kaggle/working/aoi_pass_defect_v6_multicolor_results", "zip", output_dir)
print("Download from Kaggle 'Output' tab: aoi_pass_defect_v6_multicolor_results.zip")

Download from Kaggle 'Output' tab: aoi_pass_defect_v6_multicolor_results.zip
